# Cell Type Annotation Validation

Validate CellTypist annotations with canonical marker genes.

**Contents:**
1. CellTypist prediction summary
2. Dot plots of canonical markers by cluster and annotation
3. Feature plots on UMAP for key lineage markers
4. Cross-compare CellTypist Low vs High models
5. Cluster-annotation concordance
6. Identify and investigate ambiguous clusters

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils import load_config, set_plotting_defaults
set_plotting_defaults()

cfg = load_config('../pipeline/config.yaml')
arm = 'primary'

In [ ]:
# Load annotated data
adata = sc.read_h5ad(str(Path(cfg['paths']['annotation_output']) / f'annotated_{arm}.h5ad'))
print(f"Annotated data: {adata.n_obs} cells x {adata.n_vars} genes")

# Show annotation columns
ann_cols = [c for c in adata.obs.columns if 'celltypist' in c or c in ['cell_type', 'cell_subtype']]
print(f"\nAnnotation columns: {ann_cols}")

if 'cell_type' in adata.obs.columns:
    print(f"\nCell type distribution:")
    print(adata.obs['cell_type'].value_counts())

In [ ]:
# UMAP by cell type and cluster
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sc.pl.umap(adata, color='cell_type', ax=axes[0], show=False, title='Cell type (coarse)')
sc.pl.umap(adata, color='leiden_0.8', ax=axes[1], show=False, title='Leiden clusters')
plt.tight_layout()
plt.show()

In [ ]:
# Canonical marker dot plot by cell type
markers = {
    'T cells': ['CD3D', 'CD3E'],
    'CD4 T': ['CD4', 'IL7R'],
    'CD8 T': ['CD8A', 'CD8B'],
    'NK': ['NKG7', 'GNLY', 'KLRD1'],
    'B cells': ['CD19', 'MS4A1', 'CD79A'],
    'Monocytes': ['CD14', 'LYZ', 'FCGR3A'],
    'DCs': ['FCER1A', 'CLEC10A'],
    'pDCs': ['IRF7', 'LILRA4'],
    'Platelets': ['PPBP', 'PF4'],
    'Proliferating': ['MKI67', 'TOP2A'],
}

# Flatten and check availability
all_markers = []
for genes in markers.values():
    for g in genes:
        if g in adata.var_names or (adata.raw is not None and g in adata.raw.var_names):
            if g not in all_markers:
                all_markers.append(g)

if all_markers and 'cell_type' in adata.obs.columns:
    sc.pl.dotplot(adata, var_names=all_markers, groupby='cell_type',
                  standard_scale='var', figsize=(14, 6), show=True)

In [ ]:
# Feature plots for key lineage markers
key_markers = ['CD3E', 'CD4', 'CD8A', 'MS4A1', 'CD14', 'FCGR3A', 'NKG7', 'IRF7']
available = [m for m in key_markers 
             if m in adata.var_names or (adata.raw is not None and m in adata.raw.var_names)]

if available:
    sc.pl.umap(adata, color=available, ncols=4, frameon=False, show=True)

In [ ]:
# Cluster vs annotation concordance
if 'cell_type' in adata.obs.columns:
    concordance = pd.crosstab(adata.obs['leiden_0.8'], adata.obs['cell_type'],
                               normalize='index')
    
    fig, ax = plt.subplots(figsize=(14, max(8, concordance.shape[0] * 0.4)))
    sns.heatmap(concordance, cmap='YlOrRd', annot=True, fmt='.2f', ax=ax,
                linewidths=0.5)
    ax.set_title('Cell type composition per Leiden cluster')
    ax.set_xlabel('Cell type')
    ax.set_ylabel('Leiden cluster')
    plt.tight_layout()
    plt.show()
    
    # Flag mixed clusters (no cell type >50%)
    max_frac = concordance.max(axis=1)
    mixed = max_frac[max_frac < 0.5]
    if len(mixed) > 0:
        print("Mixed clusters (no cell type >50%):")
        for cl, frac in mixed.items():
            top_types = concordance.loc[cl].nlargest(3)
            print(f"  Cluster {cl}: {', '.join([f'{t}={v:.0%}' for t, v in top_types.items()])}")

In [ ]:
# Compare Low vs High CellTypist models
low_col = [c for c in adata.obs.columns if 'celltypist' in c and 'Low' in c and 'majority' in c]
high_col = [c for c in adata.obs.columns if 'celltypist' in c and 'High' in c and 'majority' in c]

if low_col and high_col:
    low_col = low_col[0]
    high_col = high_col[0]
    
    fig, axes = plt.subplots(1, 2, figsize=(24, 8))
    sc.pl.umap(adata, color=low_col, ax=axes[0], show=False,
               title=f'CellTypist Low ({adata.obs[low_col].nunique()} types)')
    sc.pl.umap(adata, color=high_col, ax=axes[1], show=False,
               title=f'CellTypist High ({adata.obs[high_col].nunique()} types)')
    plt.tight_layout()
    plt.show()
else:
    print("Both CellTypist models not found. Available columns:", ann_cols)

In [ ]:
# Cell type proportions by HIV status
if 'cell_type' in adata.obs.columns and 'HIVstatus' in adata.obs.columns:
    props = pd.crosstab(adata.obs['HIVstatus'], adata.obs['cell_type'], normalize='index')
    
    fig, ax = plt.subplots(figsize=(12, 5))
    props.plot(kind='bar', stacked=True, ax=ax)
    ax.set_ylabel('Fraction')
    ax.set_title('Cell type proportions by HIV status')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()